
# Assignment: Multiclass Classification using CNN (MNIST)

## Objective
Build a multiclass classifier using a Convolutional Neural Network (CNN) on the MNIST handwritten digits dataset.

## Innovation Added
- EarlyStopping to prevent overfitting
- Accuracy and loss plots
- Confusion matrix
- Classification report
- Prediction confidence display
- Misclassified digit analysis


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix, classification_report


In [ ]:

# Load MNIST dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


In [ ]:

# Class names
class_names = [str(i) for i in range(10)]


In [ ]:

# Visualize sample digits
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i], cmap='gray')
    plt.title(f"Label: {y_train[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:

# Data preprocessing
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Reshape for CNN input
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

print("Processed training shape:", X_train.shape)


In [ ]:

# Build CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(10, activation='softmax')
])

model.summary()


In [ ]:

# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:

# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)


In [ ]:

# Train model
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:

# Evaluate model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy * 100, 2), "%")


In [ ]:

# Plot accuracy and loss
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:

# Predictions
y_prob = model.predict(X_test)
y_pred = np.argmax(y_prob, axis=1)


In [ ]:

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:

# Classification Report
print(classification_report(y_test, y_pred, target_names=class_names))


In [ ]:

# Display predictions with confidence
plt.figure(figsize=(12, 6))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')

    pred_class = y_pred[i]
    confidence = np.max(y_prob[i]) * 100
    true_class = y_test[i]

    color = 'green' if pred_class == true_class else 'red'

    plt.title(
        f"P: {pred_class}\n"
        f"A: {true_class}\n"
        f"{confidence:.1f}%",
        color=color,
        fontsize=8
    )
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:

# Misclassified images (Innovation)
misclassified = np.where(y_pred != y_test)[0]

print("Total misclassified images:", len(misclassified))

plt.figure(figsize=(12, 6))

for i, idx in enumerate(misclassified[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    plt.title(
        f"P: {y_pred[idx]}\nA: {y_test[idx]}",
        color='red',
        fontsize=8
    )
    plt.axis('off')

plt.tight_layout()
plt.show()



## Innovation Summary

This notebook goes beyond the basic requirements by adding:

1. EarlyStopping to reduce overfitting.
2. Accuracy and loss visualization.
3. Classification report with precision, recall, and F1-score.
4. Confidence scores for predictions.
5. Misclassified digit analysis.
